## Учёт заявок в IT-поддержке”

В компании есть небольшая IT-поддержка. Сотрудники отправляют заявки: `не работает принтер`, `забыл пароль`, `нет интернета`. Нужно сделать консольную программу, которая помогает хранить и обрабатывать эти заявки.

### Что должна уметь программа

Меню:
1. Добавить заявку
2. Показать все заявки
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
0. Выход

### Данные одной заявки
```txt
Номер заявки
Имя сотрудника
Описание проблемы
Статус
```

**Статусы можно сделать такими:**
`новая`, `в работе`, `закрыта`


### Пример заявки
```txt
№1
Сотрудник: Иван
Проблема: Не работает принтер
Статус: новая
```

### Обязательные условия
1. Номер заявки создаётся автоматически.
2. При добавлении заявки статус всегда `новая`.
3. Можно изменить статус только на:
    - `новая`
    - `в работе`
    - `закрыта`
4. При поиске по имени программа должна показывать все заявки этого сотрудника.
5. Открытые заявки — это заявки со статусом `новая` или `в работе`.
6. Решение должно быть реализовано в объектно-ориентированном стиле с соблюдением принципов `SOLID`:
   - отдельный класс для заявки;
   - отдельный класс или сервис для управления заявками;
   - отдельный класс для работы с консольным меню;
   - классы не должны брать на себя лишние обязанности;
   - зависимости между частями программы должны быть минимальными и понятными.

In [ ]:
from dataclasses import dataclass

class Status:
    new = "новая"
    in_work = "в работе"
    closed = "закрыта"

@dataclass
class Ticket:
    ticket_id: int
    name: str
    problem: str
    status: str

    def __str__(self):
        return f"№{self.ticket_id}\nСотрудник: {self.name}\nПроблема: {self.problem}\nСтатус: {self.status}\n"

class TicketManager:
    def __init__(self):
        self.tickets = []

    def add_ticket(self, employee_name: str, problem_description: str) -> Ticket:
        ticket_id = len(self.tickets) + 1
        ticket = Ticket(ticket_id, employee_name, problem_description, Status.new)
        self.tickets.append(ticket)
        return ticket

    def change_status(self, ticket_id: int, new_status: str) -> bool:
        if new_status not in [Status.new, Status.in_work, Status.closed]:
            return False
        for ticket in self.tickets:
            if ticket.ticket_id == ticket_id:
                ticket.status = new_status
                return True
        return False

    def get_all_tickets(self) -> list[Ticket]:
        return self.tickets

    def get_open_tickets(self) -> list[Ticket]:
        return [ticket for ticket in self.tickets if ticket.status in [Status.new, Status.in_work]]

    def find_tickets_by_employee(self, employee_name: str) -> list[Ticket]:
        return [ticket for ticket in self.tickets if ticket.name.lower() == employee_name.lower()]

class Menu:
    def __init__(self, ticket_manager: TicketManager):
        self.ticket_manager = ticket_manager

    def main(self):
        while True:
            print("\nМеню:")
            print("1. Добавить заявку")
            print("2. Показать все заявки")
            print("3. Изменить статус заявки")
            print("4. Показать открытые заявки")
            print("5. Найти заявку по имени сотрудника")
            print("6. Выход")

            choice = input("Выберите пункт меню: ")

            if choice == "1":
                self.add_ticket()
            elif choice == "2":
                self.show_all_tickets()
            elif choice == "3":
                self.change_ticket_status()
            elif choice == "4":
                self.show_open_tickets()
            elif choice == "5":
                self.find_tickets_by_employee()
            elif choice == "6":
                print("Выход из программы.")
                break
            else:
                print("Неверный выбор. Попробуйте снова.")

    def add_ticket(self):
        employee_name = input("Имя сотрудника: ").strip()
        problem_description = input("Описание проблемы: ").strip()
        if employee_name and problem_description:
            ticket = self.ticket_manager.add_ticket(employee_name, problem_description)
            print(f"Заявка создана: \n{ticket}")
        else:
            print("Имя сотрудника и описание проблемы не могут быть пустыми.")

    def show_all_tickets(self):
        tickets = self.ticket_manager.get_all_tickets()
        if not tickets:
            print("Нет заявок")
            return
        print("\nВсе заявки:")
        for ticket in tickets:
            print(ticket)

    def change_ticket_status(self):
        try:
            ticket_id = int(input("Номер заявки: "))
            print("Выберите статус: ")
            print("1. Новая")
            print("2. В работе")
            print("3. Закрыта")
            status_choice = input("Ваш выбор: ")

            status_map = {"1": Status.new, "2": Status.in_work, "3": Status.closed}
            if status_choice not in status_map:
                print("Неверный выбор статуса")
                return

            if self.ticket_manager.change_status(ticket_id, status_map[status_choice]):
                print(f"Статус заявки {ticket_id} обновлён")
            else:
                print(f"Ошибка: заявка с id {ticket_id} не найдена или указан неверный статус")
        except ValueError:
            print("Ошибка: номер заявки должен быть целым числом.")

    def show_open_tickets(self):
        tickets = self.ticket_manager.get_open_tickets()
        if not tickets:
            print("Нет открытых заявок")
            return
        print("\nОткрытые заявки (статус: новая или в работе):")
        for ticket in tickets:
            print(ticket)

    def find_tickets_by_employee(self):
        employee_name = input("Имя сотрудника: ").strip()
        tickets = self.ticket_manager.find_tickets_by_employee(employee_name)
        if not tickets:
            print(f"Заявок от сотрудника {employee_name} не найдено")
            return
        print(f"\nЗаявки сотрудника {employee_name}:")
        for ticket in tickets:
            print(ticket)

if __name__ == "__main__":
    ticket_manager = TicketManager()
    menu = Menu(ticket_manager)
    menu.main()







Меню:
1. Добавить заявку
2. Показать все заявки
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход
Заявка создана: Ticket(ticket_id=1, name='fjdskhfjf', problem='gfjhfkjhf', status='новая')

Меню:
1. Добавить заявку
2. Показать все заявки
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход
Ticket(ticket_id=1, name='fjdskhfjf', problem='gfjhfkjhf', status='новая')

Меню:
1. Добавить заявку
2. Показать все заявки
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход
Выберите статус: 
1. Новая
2. В работе
3. Закрыта


AttributeError: 'TicketManager' object has no attribute 'update_ticket_status'